In [ ]:
import itertools
import logging
import os

import mlflow

from model.train_models import train_evaluate_model
from utils.data_prep import get_clean_combined_data

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

mlflow.sklearn.autolog(disable=True)

In [ ]:
completed_runs_file = "completed_runs.txt"

if os.path.exists(completed_runs_file):
    with open(completed_runs_file, "r") as f:
        completed_runs = {line.strip() for line in f if line.strip()}
else:
    completed_runs = set()

print(f"Loaded {len(completed_runs)} completed runs from memory.")

In [ ]:
# ks = [0.25, 0.5, 0.75, 1]
# ns = [4, 5]
# event_cols = ["sub_event_type", "event_type"]
# remove_abyei_options = [False]  # TODO Abyei currently not present for rainfall
# include_food_options = [True, False]
# include_rain_options = [True, False]
# include_text_options = [True, False]

In [ ]:
xgb_params = {
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 3, 5],
    "max_delta_step": [0, 1, 5],
    "gamma": [0, 1, 3, 5],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0, 0.1, 1, 2],
    "reg_lambda": [1, 5, 10],
    "colsample_bylevel": [0.6, 0.8, 1.0],
}

In [ ]:
# data_configs = itertools.product(
#     remove_abyei_options,
#     include_food_options,
#     include_rain_options,
#     include_text_options,
#     ks,
#     event_cols,
# )

# for (
#     remove_abyei,
#     include_food,
#     include_rain,
#     include_text,
#     k,
#     event_col,
# ) in data_configs:
#     food_str = "_food" if include_food else ""
#     rain_str = "_rain" if include_rain else ""
#     text_str = "_text" if include_text else ""
#     abyei_str = "_remove_abyei" if remove_abyei else ""
#     event_str = "event" if event_col == "event_type" else "sub"

#     which_data = f"acled_{event_str}{food_str}{rain_str}{text_str}{abyei_str}"

#     if include_text:
#         pca_options = [
#             True
#         ]  # Run both when text is included #TODO this is temp changed !!!!!!!!!!!!!
#     else:
#         pca_options = [False]  # Only run without PCA when text isn't included

#     all_runs_completed = True
#     for n in ns:
#         for use_pca in pca_options:
#             pca_str = "_pca" if use_pca else ""
#             expected_run = f"{which_data}{pca_str}_{k}_{n}"
#             if expected_run not in completed_runs:
#                 all_runs_completed = False
#                 break  # Stop checking this inner loop if we find a missing run
#         if not all_runs_completed:
#             break  # Stop checking the outer loop too

#     if all_runs_completed:
#         print(
#             f"Skipping data load for {which_data} - all associated runs are complete."
#         )
#         continue

#     # Only load data if I have at least one missing run
#     data_sources = [
#         src
#         for src, include in zip(
#             ["food", "rain", "text"], [include_food, include_rain, include_text]
#         )
#         if include
#     ]

#     model_data, predictor_cols = get_clean_combined_data(
#         data_sources=data_sources,
#         download=DOWNLOAD,
#         remove_abyei=remove_abyei,
#         k=k,
#         event_col=event_col,
#     )

#     for n in ns:
#         for use_pca in pca_options:
#             pca_str = "_pca" if use_pca else ""
#             run_name = f"{which_data}{pca_str}_{k}_{n}"

#             if run_name in completed_runs:
#                 print(f"Skipping already completed run: {run_name}")
#                 continue

#             all_params = {
#                 **xgb_params,
#                 "k": k,
#                 "event_col": event_col,
#                 "remove_abyei": remove_abyei,
#                 "n_splits": n,
#                 "use_pca": use_pca,
#             }

#             with mlflow.start_run(run_name=run_name):
#                 mlflow.set_tags(
#                     {
#                         "data_version": which_data,
#                         "remove_abyei": remove_abyei,
#                         "include_food": include_food,
#                         "include_rain": include_rain,
#                         "include_text": include_text,
#                         "use_pca": use_pca,
#                         "k": k,
#                         "n_splits": n,
#                         "event_col": event_col,
#                     }
#                 )
#                 logger.info(f"Running mode: {run_name}")

#                 results, best_params = train_evaluate_model(
#                     model_data,
#                     predictor_cols,
#                     all_params,
#                     best_params=False,
#                     use_pca=use_pca,
#                 )

#                 mlflow.log_params(best_params)
#                 mlflow.log_metrics({key: float(val) for key, val in results.items()})
#                 mlflow.log_dict(results, "model_report.json")

#                 completed_runs.add(run_name)  # Add to log file
#                 with open(completed_runs_file, "a") as f:
#                     f.write(run_name + "\n")

In [ ]:
ks = [0.25, 0.5, 0.75, 1]
ns = [4, 5]
event_cols = ["sub_event_type", "event_type"]
include_food_options = [True, False]
include_rain_options = [True, False]
include_text_options = [True, False]
conflict_only_embedding_options = [True, False]  # Only used when include_text is True

In [ ]:
data_configs = itertools.product(
    include_food_options,
    include_rain_options,
    include_text_options,
    ks,
    event_cols,
)

for (
    include_food,
    include_rain,
    include_text,
    k,
    event_col,
) in data_configs:
    food_str = "_food" if include_food else ""
    rain_str = "_rain" if include_rain else ""
    event_str = "event" if event_col == "event_type" else "sub"

    if include_text:
        pca_options = [True, False]
        conflict_only_options = conflict_only_embedding_options
    else:
        pca_options = [False]  # Only run without PCA when text isn't included
        conflict_only_options = [None]  # Not applicable when text isn't included

    for conflict_only in conflict_only_options:
        if include_text:
            text_str = "_text_conflict" if conflict_only else "_text_all"
        else:
            text_str = ""

        which_data = f"acled_{event_str}{food_str}{rain_str}{text_str}"

        all_runs_completed = True
        for n in ns:
            for use_pca in pca_options:
                pca_str = "_pca" if use_pca else ""
                expected_run = f"{which_data}{pca_str}_{k}_{n}"
                if expected_run not in completed_runs:
                    all_runs_completed = False
                    break  # Stop checking this inner loop if we find a missing run
            if not all_runs_completed:
                break  # Stop checking the outer loop too

        if all_runs_completed:
            print(
                f"Skipping data load for {which_data} - all associated runs are complete."
            )
            continue

        # Only load data if I have at least one missing run
        data_sources = [
            src
            for src, include in zip(
                ["food", "rain", "text"], [include_food, include_rain, include_text]
            )
            if include
        ]

        model_data, predictor_cols = get_clean_combined_data(
            data_sources=data_sources,
            k=k,
            event_col=event_col,
            conflict_only_embeddings=bool(conflict_only),
        )

        for n in ns:
            for use_pca in pca_options:
                pca_str = "_pca" if use_pca else ""
                run_name = f"{which_data}{pca_str}_{k}_{n}"

                if run_name in completed_runs:
                    print(f"Skipping already completed run: {run_name}")
                    continue

                all_params = {
                    **xgb_params,
                    "k": k,
                    "event_col": event_col,
                    "n_splits": n,
                    "use_pca": use_pca,
                }

                with mlflow.start_run(run_name=run_name):
                    mlflow.set_tags(
                        {
                            "data_version": which_data,
                            "remove_abyei": True,
                            "include_food": include_food,
                            "include_rain": include_rain,
                            "include_text": include_text,
                            "conflict_only_embeddings": bool(conflict_only),
                            "use_pca": use_pca,
                            "k": k,
                            "n_splits": n,
                            "event_col": event_col,
                        }
                    )
                    logger.info(f"Running mode: {run_name}")

                    results, best_params = train_evaluate_model(
                        model_data,
                        predictor_cols,
                        all_params,
                        best_params=False,
                        use_pca=use_pca,
                    )

                    mlflow.log_params(best_params)
                    mlflow.log_metrics({key: float(val) for key, val in results.items()})
                    mlflow.log_dict(results, "model_report.json")

                    completed_runs.add(run_name)  # Add to log file
                    with open(completed_runs_file, "a") as f:
                        f.write(run_name + "\n")